# Time series + rolling windows
pandas contains extensive capabilities and features for working with time series data for all domains. Using the NumPy `datetime64` and `timedelta64` dtypes, pandas has consolidated a large number of features from other Python libraries like `scikits.timeseries` as well as created a tremendous amount of new functionality for manipulating time series data.

In [1]:
# Parsing time series information from various sources and formats
import pandas as pd
import numpy as np
import datetime

dti = pd.to_datetime(
    ['1/1/2018', np.datetime64('2018-01-01'), datetime.datetime(2018, 1, 1)]
)
dti

DatetimeIndex(['2018-01-01', '2018-01-01', '2018-01-01'], dtype='datetime64[us]', freq=None)

In [2]:
# Generate sequences of fixed-frequency dates and time spans
dti = pd.date_range('2018-01-01', periods=3, freq='h')
dti

DatetimeIndex(['2018-01-01 00:00:00', '2018-01-01 01:00:00',
               '2018-01-01 02:00:00'],
              dtype='datetime64[us]', freq='h')

In [3]:
# Manipulating and converting date times with timezone information
dti = dti.tz_localize("UTC")
dti

DatetimeIndex(['2018-01-01 00:00:00+00:00', '2018-01-01 01:00:00+00:00',
               '2018-01-01 02:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq='h')

In [4]:
dti.tz_convert("US/Pacific")

DatetimeIndex(['2017-12-31 16:00:00-08:00', '2017-12-31 17:00:00-08:00',
               '2017-12-31 18:00:00-08:00'],
              dtype='datetime64[us, US/Pacific]', freq='h')

In [5]:
# Resampling or converting a time series to a particular frequency
idx = pd.date_range("2018-01-01", periods=5, freq="h")
ts = pd.Series(range(len(idx)), index=idx)
ts

2018-01-01 00:00:00    0
2018-01-01 01:00:00    1
2018-01-01 02:00:00    2
2018-01-01 03:00:00    3
2018-01-01 04:00:00    4
Freq: h, dtype: int64

In [6]:
ts.resample("2h").mean()

2018-01-01 00:00:00    0.5
2018-01-01 02:00:00    2.5
2018-01-01 04:00:00    4.0
Freq: 2h, dtype: float64

In [7]:
# Performing date and time arithmetic with absolute or relative time increments
friday = pd.Timestamp("2018-01-05")
friday.day_name()

'Friday'

In [8]:
#Add 1 day
saturday = friday + pd.Timedelta('1 day')
saturday.day_name()

'Saturday'

In [9]:
# Add 1 business day (Friday --> Monday)
monday = friday + pd.offsets.BDay()
monday.day_name()

'Monday'

### Overview

pandas captures 4 general time related concepts:

1. Date times: A specific date and time with timezone support. Similar to `datetime.datetime` from the standard library.
2. Time deltas: An absolute time duration. Similar to `datetime.timedelta` from the standard library.
3. Time spans: A span of time defined by a point in time and its associated frequency.
4. Date offsets: A relative time duration that respects calendar arithmetic. Similar to `dateutil.relativedelta.relativedelta` from the `dateutil` package.

| Concept      | Scalar Class | Array Class      | pandas Data Type                         | Primary Creation Method             |
| ------------ | ------------ | ---------------- | ---------------------------------------- | ----------------------------------- |
| Date times   | `Timestamp`  | `DatetimeIndex`  | `datetime64[ns]` or `datetime64[ns, tz]` | `to_datetime` or `date_range`       |
| Time deltas  | `Timedelta`  | `TimedeltaIndex` | `timedelta64[ns]`                        | `to_timedelta` or `timedelta_range` |
| Time spans   | `Period`     | `PeriodIndex`    | `period[freq]`                           | `Period` or `period_range`          |
| Date offsets | `DateOffset` | `None`           | `None`                                   | `DateOffset`                        |

For time series data, it’s conventional to represent the time component in the index of a **`Series`** or **`DataFrame`** so manipulations can be performed with respect to the time element.


In [10]:
pd.Series(range(3), index = pd.date_range("2000", freq = 'D', periods = 3))

2000-01-01    0
2000-01-02    1
2000-01-03    2
Freq: D, dtype: int64

In [11]:
# However, Series and DataFrame can directly also support the time component as data itself.
pd.Series(pd.date_range('2000', freq = 'D', periods = 3))

0   2000-01-01
1   2000-01-02
2   2000-01-03
dtype: datetime64[us]

`Series` and `DataFrame` have extended data type support and functionality for `datetime`, `timedelta` and `Period` data when passed into those constructors. `DateOffset` data however will be stored as `object` data.

In [12]:
pd.Series(pd.period_range('1/1/2011', freq = 'M', periods = 3))

0    2011-01
1    2011-02
2    2011-03
dtype: period[M]

In [13]:
pd.Series([pd.DateOffset(1), pd.DateOffset(2)])

0         <DateOffset>
1    <2 * DateOffsets>
dtype: object

In [14]:
pd.Series(pd.date_range('1/1/2011', freq= 'ME', periods = 3))

0   2011-01-31
1   2011-02-28
2   2011-03-31
dtype: datetime64[us]

Lastly, pandas represents null date times, time deltas, and time spans as `NaT` which is useful for representing missing or null date like values and behaves similar as `np.nan` does for float data.

In [15]:
pd.Timestamp(pd.NaT)

NaT

In [16]:
pd.Timedelta(pd.NaT)

NaT

In [17]:
pd.Period(pd.NaT)

NaT

In [18]:
# Equality acts as np.nan would
pd.NaT == pd.NaT

False

### Timestamps vs. time spans

In [19]:
# Timestamped data is the most basic type of time series data that associates values with points in time. 
# For pandas objects it means using the points in time.

import datetime
pd.Timestamp(datetime.datetime(2012, 5, 1))

Timestamp('2012-05-01 00:00:00')

In [20]:
pd.Timestamp('2012-05-01')

Timestamp('2012-05-01 00:00:00')

In [21]:
pd.Timestamp(2012, 5, 1)

Timestamp('2012-05-01 00:00:00')

However, in many cases it is more natural to associate things like change variables with a time span instead. The span represented by `Period` can be specified explicitly, or inferred from datetime string format.

For example:

In [22]:
pd.Period('2011-01')

Period('2011-01', 'M')

In [23]:
pd.Period('2012-05', freq = 'D')

Period('2012-05-01', 'D')

`Timestamp` and `Period` can serve as an index. Lists of `Timestamp` and `Period` are automatically coerced to `DatetimeIndex` and `PeriodIndex` respectively.

In [24]:
dates = [
    pd.Timestamp("2012-05-01"),
    pd.Timestamp("2012-05-02"),
    pd.Timestamp("2012-05-03"),
]


In [25]:
ts = pd.Series(np.random.randn(3), dates)

In [26]:
type (ts.index)

pandas.DatetimeIndex

In [27]:
ts.index

DatetimeIndex(['2012-05-01', '2012-05-02', '2012-05-03'], dtype='datetime64[us]', freq=None)

In [28]:
ts

2012-05-01    1.754525
2012-05-02    0.216684
2012-05-03    0.138879
dtype: float64

In [29]:
periods = [pd.Period("2012-01"), pd.Period("2012-02"), pd.Period("2012-03")]

In [30]:
ts = pd.Series(np.random.randn(3), periods)

In [31]:
type(ts.index)

pandas.PeriodIndex

In [32]:
ts

2012-01   -0.027817
2012-02    1.317074
2012-03    0.039124
Freq: M, dtype: float64

pandas allows you to capture both representations and convert between them. Under the hood, pandas represents timestamps using instances of `Timestamp` and sequences of timestamps using instances of `DatetimeIndex`. For regular time spans, pandas uses `Period` objects for scalar values and `PeriodIndex` for sequences of spans. Better support for irregular intervals with arbitrary start and end points are forth-coming in future releases


### Converting to timestamps

To convert a **`Series`** or list-like object of date-like objects e.g. strings, epochs, or a mixture, you can use the `to_datetime` function. When passed a **`Series`**, this returns a **`Series`** (with the same index), while a list-like is converted to a **`DatetimeIndex`**:


In [33]:
pd.to_datetime(pd.Series(['Jul 31, 2009', 'Jan 10, 2010', None]))

0   2009-07-31
1   2010-01-10
2          NaT
dtype: datetime64[us]

In [34]:
pd.to_datetime(['2005/11/23', '2010/12/31'])

DatetimeIndex(['2005-11-23', '2010-12-31'], dtype='datetime64[us]', freq=None)

In [35]:
# If you use dates which start with the day first (i.e. European style), you can pass the dayfirst flag:
pd.to_datetime(["04-01-2012 10:00"], dayfirst=True)

DatetimeIndex(['2012-01-04 10:00:00'], dtype='datetime64[us]', freq=None)

In [36]:
pd.to_datetime(["04-14-2012 10:00"], dayfirst=True)

/tmp/ipykernel_1622/2353669186.py:1: UserWarning: Parsing dates in %m-%d-%Y %H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(["04-14-2012 10:00"], dayfirst=True)


DatetimeIndex(['2012-04-14 10:00:00'], dtype='datetime64[us]', freq=None)

If you pass a single string to `to_datetime`, it returns a single `Timestamp`. `Timestamp` can also accept string input, but it doesn’t accept string parsing options like `dayfirst` or `format`, so use `to_datetime` if these are required.

In [37]:
pd.to_datetime('2010/11/12')

Timestamp('2010-11-12 00:00:00')

In [38]:
pd.Timestamp('2010/11/12')

Timestamp('2010-11-12 00:00:00')

In [39]:
# You can also use the DatetimeIndex constructor directly:
pd.DatetimeIndex(["2018-01-01", "2018-01-03", "2018-01-05"])

DatetimeIndex(['2018-01-01', '2018-01-03', '2018-01-05'], dtype='datetime64[us]', freq=None)

In [40]:
# The string ‘infer’ can be passed in order to set the frequency of the index as the inferred frequency upon creation:
pd.DatetimeIndex(["2018-01-01", "2018-01-03", "2018-01-05"], freq="infer")

DatetimeIndex(['2018-01-01', '2018-01-03', '2018-01-05'], dtype='datetime64[us]', freq='2D')

In most cases, parsing strings to datetimes (with any of to_datetime(), DatetimeIndex, or Timestamp) will produce objects with microsecond (“us”) unit. The exception to this rule is if your strings have nanosecond precision, in which case the result will have “ns” unit:

In [41]:
pd.to_datetime(["2016-01-01 02:03:04"]).unit

'us'

In [42]:
pd.to_datetime(["2016-01-01 02:03:04.123"]).unit

'us'

In [43]:
pd.to_datetime(["2016-01-01 02:03:04.123456"]).unit

'us'

In [44]:
pd.to_datetime(["2016-01-01 02:03:04.123456789"]).unit

'ns'

### Providing a format argument
In addition to the required datetime string, a `format` argument can be passed to ensure specific parsing. This could also potentially speed up the conversion considerably.

In [45]:
pd.to_datetime('2010/11/12', format="%Y/%m/%d")

Timestamp('2010-11-12 00:00:00')

In [46]:
pd.to_datetime("12-11-2010 00:00", format="%d-%m-%Y %H:%M")

Timestamp('2010-11-12 00:00:00')

### Invalid data
The default behavior, `errors='raise'`, is to raise when unparsable:

In [47]:
pd.to_datetime(['2009/07/31', 'asd'], errors='raise')

ValueError: time data "asd" doesn't match format "%Y/%m/%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [48]:
# Pass errors='coerce' to convert unparsable data to NaT (not a time):
pd.to_datetime(["2009/07/31", "asd"], errors="coerce")

DatetimeIndex(['2009-07-31', 'NaT'], dtype='datetime64[us]', freq=None)

### Generating ranges of timestamps
To generate an index with timestamps, you can use either the DatetimeIndex or Index constructor and pass in a list of datetime objects:

In [49]:
dates = [
    datetime.datetime(2012, 5, 1),
    datetime.datetime(2012, 5, 2),
    datetime.datetime(2012, 5, 3),
]
# Note the frequency information
index = pd.DatetimeIndex(dates)

In [50]:
index

DatetimeIndex(['2012-05-01', '2012-05-02', '2012-05-03'], dtype='datetime64[us]', freq=None)

In [51]:
# Automatically converted to DatetimeIndex
index = pd.Index(dates)

In [52]:
index

DatetimeIndex(['2012-05-01', '2012-05-02', '2012-05-03'], dtype='datetime64[us]', freq=None)

In practice this becomes very cumbersome because we often need a very long index with a large number of timestamps. If we need timestamps on a regular frequency, we can use the **`date_range()`** and **`bdate_range()`** functions to create a `DatetimeIndex`. The default frequency for `date_range` is a **calendar day** while the default for `bdate_range` is a **business day**:


In [53]:
start = datetime.datetime(2011, 1, 1)
end = datetime.datetime(2012, 1, 1)
index = pd.date_range(start, end)
index

DatetimeIndex(['2011-01-01', '2011-01-02', '2011-01-03', '2011-01-04',
               '2011-01-05', '2011-01-06', '2011-01-07', '2011-01-08',
               '2011-01-09', '2011-01-10',
               ...
               '2011-12-23', '2011-12-24', '2011-12-25', '2011-12-26',
               '2011-12-27', '2011-12-28', '2011-12-29', '2011-12-30',
               '2011-12-31', '2012-01-01'],
              dtype='datetime64[us]', length=366, freq='D')

In [54]:
index = pd.bdate_range(start, end)
index

DatetimeIndex(['2011-01-03', '2011-01-04', '2011-01-05', '2011-01-06',
               '2011-01-07', '2011-01-10', '2011-01-11', '2011-01-12',
               '2011-01-13', '2011-01-14',
               ...
               '2011-12-19', '2011-12-20', '2011-12-21', '2011-12-22',
               '2011-12-23', '2011-12-26', '2011-12-27', '2011-12-28',
               '2011-12-29', '2011-12-30'],
              dtype='datetime64[us]', length=260, freq='B')

In [55]:
# Convenience functions like date_range and bdate_range can utilize a variety of frequency aliases:
pd.date_range(start, periods = 1000, freq = 'ME')

DatetimeIndex(['2011-01-31', '2011-02-28', '2011-03-31', '2011-04-30',
               '2011-05-31', '2011-06-30', '2011-07-31', '2011-08-31',
               '2011-09-30', '2011-10-31',
               ...
               '2093-07-31', '2093-08-31', '2093-09-30', '2093-10-31',
               '2093-11-30', '2093-12-31', '2094-01-31', '2094-02-28',
               '2094-03-31', '2094-04-30'],
              dtype='datetime64[us]', length=1000, freq='ME')

In [56]:
pd.bdate_range(start, periods = 250, freq = 'BQS')

DatetimeIndex(['2011-01-03', '2011-04-01', '2011-07-01', '2011-10-03',
               '2012-01-02', '2012-04-02', '2012-07-02', '2012-10-01',
               '2013-01-01', '2013-04-01',
               ...
               '2071-01-01', '2071-04-01', '2071-07-01', '2071-10-01',
               '2072-01-01', '2072-04-01', '2072-07-01', '2072-10-03',
               '2073-01-02', '2073-04-03'],
              dtype='datetime64[us]', length=250, freq='BQS-JAN')

`date_range` and `bdate_range` make it easy to generate a range of dates using various combinations of parameters like `start`, `end`, `periods`, and `freq`. The start and end dates are strictly inclusive, so dates outside of those specified will not be generated:

In [57]:
pd.date_range(start, end, freq = 'BME')

DatetimeIndex(['2011-01-31', '2011-02-28', '2011-03-31', '2011-04-29',
               '2011-05-31', '2011-06-30', '2011-07-29', '2011-08-31',
               '2011-09-30', '2011-10-31', '2011-11-30', '2011-12-30'],
              dtype='datetime64[us]', freq='BME')

In [58]:
pd.date_range(start, end, freq = 'W')

DatetimeIndex(['2011-01-02', '2011-01-09', '2011-01-16', '2011-01-23',
               '2011-01-30', '2011-02-06', '2011-02-13', '2011-02-20',
               '2011-02-27', '2011-03-06', '2011-03-13', '2011-03-20',
               '2011-03-27', '2011-04-03', '2011-04-10', '2011-04-17',
               '2011-04-24', '2011-05-01', '2011-05-08', '2011-05-15',
               '2011-05-22', '2011-05-29', '2011-06-05', '2011-06-12',
               '2011-06-19', '2011-06-26', '2011-07-03', '2011-07-10',
               '2011-07-17', '2011-07-24', '2011-07-31', '2011-08-07',
               '2011-08-14', '2011-08-21', '2011-08-28', '2011-09-04',
               '2011-09-11', '2011-09-18', '2011-09-25', '2011-10-02',
               '2011-10-09', '2011-10-16', '2011-10-23', '2011-10-30',
               '2011-11-06', '2011-11-13', '2011-11-20', '2011-11-27',
               '2011-12-04', '2011-12-11', '2011-12-18', '2011-12-25',
               '2012-01-01'],
              dtype='datetime64[us]', freq='W-S

In [59]:
pd.bdate_range(end = end, periods = 20)

DatetimeIndex(['2011-12-06', '2011-12-07', '2011-12-08', '2011-12-09',
               '2011-12-12', '2011-12-13', '2011-12-14', '2011-12-15',
               '2011-12-16', '2011-12-19', '2011-12-20', '2011-12-21',
               '2011-12-22', '2011-12-23', '2011-12-26', '2011-12-27',
               '2011-12-28', '2011-12-29', '2011-12-30'],
              dtype='datetime64[us]', freq='B')

In [60]:
pd.bdate_range(start = start, periods = 20)

DatetimeIndex(['2011-01-03', '2011-01-04', '2011-01-05', '2011-01-06',
               '2011-01-07', '2011-01-10', '2011-01-11', '2011-01-12',
               '2011-01-13', '2011-01-14', '2011-01-17', '2011-01-18',
               '2011-01-19', '2011-01-20', '2011-01-21', '2011-01-24',
               '2011-01-25', '2011-01-26', '2011-01-27', '2011-01-28'],
              dtype='datetime64[us]', freq='B')

In [61]:
# Specifying start, end, and periods will generate a range of evenly spaced dates from start 
# to end inclusively, with periods number of elements in the resulting DatetimeIndex:

pd.date_range('2018-01-01', '2018-01-05', periods = 5)

DatetimeIndex(['2018-01-01', '2018-01-02', '2018-01-03', '2018-01-04',
               '2018-01-05'],
              dtype='datetime64[us]', freq=None)

In [62]:
pd.date_range("2018-01-01", "2018-01-05", periods=10)

DatetimeIndex(['2018-01-01 00:00:00', '2018-01-01 10:40:00',
               '2018-01-01 21:20:00', '2018-01-02 08:00:00',
               '2018-01-02 18:40:00', '2018-01-03 05:20:00',
               '2018-01-03 16:00:00', '2018-01-04 02:40:00',
               '2018-01-04 13:20:00', '2018-01-05 00:00:00'],
              dtype='datetime64[us]', freq=None)

## Indexing

One of the main uses for `DatetimeIndex` is as an index for pandas objects. The `DatetimeIndex` class contains many time series related optimizations:

* A large range of dates for various offsets are pre-computed and cached under the hood in order to make generating subsequent date ranges very fast (just have to grab a slice).
* Fast shifting using the `shift` method on pandas objects.
* Unioning of overlapping `DatetimeIndex` objects with the same frequency is very fast (important for fast data alignment).
* Quick access to date fields via properties such as `year`, `month`, etc.
* Regularization functions like `snap` and very fast `asof` logic.

`DatetimeIndex` objects have all the basic functionality of regular `Index` objects, and a smorgasbord of advanced time series specific methods for easy frequency processing.

**Note**

While pandas does not force you to have a sorted date index, some of these methods may have unexpected or incorrect behavior if the dates are unsorted.

`DatetimeIndex` can be used like a regular index and offers all of its intelligent functionality like selection, slicing, etc.


In [63]:
rng = pd.date_range(start, end, freq = 'BME')
ts = pd.Series(np.random.randn(len(rng)), index = rng)
ts.index

DatetimeIndex(['2011-01-31', '2011-02-28', '2011-03-31', '2011-04-29',
               '2011-05-31', '2011-06-30', '2011-07-29', '2011-08-31',
               '2011-09-30', '2011-10-31', '2011-11-30', '2011-12-30'],
              dtype='datetime64[us]', freq='BME')

In [64]:
ts[:5].index

DatetimeIndex(['2011-01-31', '2011-02-28', '2011-03-31', '2011-04-29',
               '2011-05-31'],
              dtype='datetime64[us]', freq='BME')

In [65]:
ts[::2].index

DatetimeIndex(['2011-01-31', '2011-03-31', '2011-05-31', '2011-07-29',
               '2011-09-30', '2011-11-30'],
              dtype='datetime64[us]', freq='2BME')

### Partial string indexing

In [66]:
# Dates and strings that parse to timestamps can be passed as indexing parameters:
ts["1/31/2011"]

np.float64(0.889911951181064)

In [67]:
ts[datetime.datetime(2011, 12, 25):]

2011-12-30   -0.938022
Freq: BME, dtype: float64

In [68]:
ts["10/31/2011":"12/31/2011"]

2011-10-31   -0.246905
2011-11-30   -1.673974
2011-12-30   -0.938022
Freq: BME, dtype: float64

In [69]:
# To provide convenience for accessing longer time series, you can also pass in the year or year and month as strings:
ts['2011']

2011-01-31    0.889912
2011-02-28    0.288308
2011-03-31    0.862329
2011-04-29    0.724004
2011-05-31    1.087400
2011-06-30    0.314477
2011-07-29   -1.123341
2011-08-31    0.484211
2011-09-30   -0.284718
2011-10-31   -0.246905
2011-11-30   -1.673974
2011-12-30   -0.938022
Freq: BME, dtype: float64

In [70]:
ts['2011-6']

2011-06-30    0.314477
Freq: BME, dtype: float64

This type of slicing will work on a `DataFrame` with a `DatetimeIndex` as well. Since the partial string selection is a form of label slicing, the endpoints **will be** included. This would include matching times on an included date:

##### ⚠️ Warning

Indexing `DataFrame` rows with a *single* string with getitem (e.g. `frame[dtstring]`) is deprecated starting with pandas 1.2.0 (given the ambiguity whether it is indexing the rows or selecting a column) and will be removed in a future version. The equivalent with `.loc` (e.g. `frame.loc[dtstring]`) is still supported.


In [71]:
dft = pd.DataFrame(
    np.random.randn(100000, 1),
    columns=["A"],
    index=pd.date_range("20130101", periods=100000, freq="min"),
)
dft

,A
2013-01-01 00:00:00,-0.265494
2013-01-01 00:01:00,1.613487
2013-01-01 00:02:00,-0.124030
2013-01-01 00:03:00,1.683153
2013-01-01 00:04:00,-0.211354
...,...
2013-03-11 10:35:00,0.482034
2013-03-11 10:36:00,0.000302
2013-03-11 10:37:00,0.076287
2013-03-11 10:38:00,0.571138


In [72]:
dft.loc['2013']

,A
2013-01-01 00:00:00,-0.265494
2013-01-01 00:01:00,1.613487
2013-01-01 00:02:00,-0.124030
2013-01-01 00:03:00,1.683153
2013-01-01 00:04:00,-0.211354
...,...
2013-03-11 10:35:00,0.482034
2013-03-11 10:36:00,0.000302
2013-03-11 10:37:00,0.076287
2013-03-11 10:38:00,0.571138


In [73]:
# This starts on the very first time in the month, and includes the last date and time for the month:
dft["2013-1":"2013-2"]

,A
2013-01-01 00:00:00,-0.265494
2013-01-01 00:01:00,1.613487
2013-01-01 00:02:00,-0.124030
2013-01-01 00:03:00,1.683153
2013-01-01 00:04:00,-0.211354
...,...
2013-02-28 23:55:00,-0.255568
2013-02-28 23:56:00,0.016835
2013-02-28 23:57:00,-0.094081
2013-02-28 23:58:00,0.685120


In [74]:
# This specifies a stop time that includes all of the times on the last day
dft["2013-1":"2013-2-28"]

,A
2013-01-01 00:00:00,-0.265494
2013-01-01 00:01:00,1.613487
2013-01-01 00:02:00,-0.124030
2013-01-01 00:03:00,1.683153
2013-01-01 00:04:00,-0.211354
...,...
2013-02-28 23:55:00,-0.255568
2013-02-28 23:56:00,0.016835
2013-02-28 23:57:00,-0.094081
2013-02-28 23:58:00,0.685120


In [75]:
# This specifies an exact stop time (and is not the same as the above):
dft["2013-1":"2013-2-28 00:00:00"]

,A
2013-01-01 00:00:00,-0.265494
2013-01-01 00:01:00,1.613487
2013-01-01 00:02:00,-0.124030
2013-01-01 00:03:00,1.683153
2013-01-01 00:04:00,-0.211354
...,...
2013-02-27 23:56:00,-0.893035
2013-02-27 23:57:00,0.235663
2013-02-27 23:58:00,-0.098454
2013-02-27 23:59:00,0.455375


In [76]:
# We are stopping on the included end-point as it is part of the index:
dft["2013-1-15":"2013-1-15 12:30:00"]

,A
2013-01-15 00:00:00,-0.340058
2013-01-15 00:01:00,-0.414573
2013-01-15 00:02:00,-1.625333
2013-01-15 00:03:00,-0.644704
2013-01-15 00:04:00,0.517272
...,...
2013-01-15 12:26:00,0.891595
2013-01-15 12:27:00,0.943890
2013-01-15 12:28:00,-1.163305
2013-01-15 12:29:00,0.271676


`DatetimeIndex` partial string indexing also works on a `DataFrame` with a `MultiIndex`:

In [77]:
dft2 = pd.DataFrame(
    np.random.randn(20, 1),
    columns=["A"],
    index=pd.MultiIndex.from_product(
        [pd.date_range("20130101", periods=10, freq="12h"), ["a", "b"]]
    ),
)

dft2

A
2013-01-01 00:00:00 a  1.320327
                    b -1.243453
2013-01-01 12:00:00 a  1.578206
                    b  0.579819
2013-01-02 00:00:00 a -1.394773
                    b -0.209511
2013-01-02 12:00:00 a  2.106614
                    b  0.283503
2013-01-03 00:00:00 a -0.019715
                    b  0.204092
2013-01-03 12:00:00 a -1.393033
                    b  0.370692
2013-01-04 00:00:00 a -0.607319
                    b  1.101380
2013-01-04 12:00:00 a  1.103174
                    b  2.184603
2013-01-05 00:00:00 a -2.406266
                    b  0.686918
2013-01-05 12:00:00 a  0.553147
                    b  0.310908

In [78]:
dft2.loc["2013-01-05"]

A
2013-01-05 00:00:00 a -2.406266
                    b  0.686918
2013-01-05 12:00:00 a  0.553147
                    b  0.310908

In [79]:
idx = pd.IndexSlice
dft2 = dft2.swaplevel(0, 1).sort_index()
dft2.loc[idx[:, "2013-01-05"], :]

A
a 2013-01-05 00:00:00 -2.406266
  2013-01-05 12:00:00  0.553147
b 2013-01-05 00:00:00  0.686918
  2013-01-05 12:00:00  0.310908

In [80]:
# Slicing with string indexing also honors UTC offset.
df = pd.DataFrame([0], index=pd.DatetimeIndex(["2019-01-01"], tz="US/Pacific"))
df

,0
2019-01-01 00:00:00-08:00,0


In [81]:
df["2019-01-01 12:00:00+04:00":"2019-01-01 13:00:00+04:00"]


,0
2019-01-01 00:00:00-08:00,0


## Time/date components

There are several time/date properties that one can access from `Timestamp` or a collection of timestamps like a `DatetimeIndex`.

| Property           | Description                                                       |
| ------------------ | ----------------------------------------------------------------- |
| `year`             | The year of the datetime                                          |
| `month`            | The month of the datetime                                         |
| `day`              | The days of the datetime                                          |
| `hour`             | The hour of the datetime                                          |
| `minute`           | The minutes of the datetime                                       |
| `second`           | The seconds of the datetime                                       |
| `microsecond`      | The microseconds of the datetime                                  |
| `nanosecond`       | The nanoseconds of the datetime                                   |
| `date`             | Returns datetime.date (does not contain timezone information)     |
| `time`             | Returns datetime.time (does not contain timezone information)     |
| `timetz`           | Returns datetime.time as local time with timezone information     |
| `dayofyear`        | The ordinal day of year                                           |
| `day_of_year`      | The ordinal day of year                                           |
| `dayofweek`        | The number of the day of the week with Monday=0, Sunday=6         |
| `day_of_week`      | The number of the day of the week with Monday=0, Sunday=6         |
| `weekday`          | The number of the day of the week with Monday=0, Sunday=6         |
| `quarter`          | Quarter of the date: Jan-Mar = 1, Apr-Jun = 2, etc.               |
| `days_in_month`    | The number of days in the month of the datetime                   |
| `is_month_start`   | Logical indicating if first day of month (defined by frequency)   |
| `is_month_end`     | Logical indicating if last day of month (defined by frequency)    |
| `is_quarter_start` | Logical indicating if first day of quarter (defined by frequency) |
| `is_quarter_end`   | Logical indicating if last day of quarter (defined by frequency)  |
| `is_year_start`    | Logical indicating if first day of year (defined by frequency)    |
| `is_year_end`      | Logical indicating if last day of year (defined by frequency)     |
| `is_leap_year`     | Logical indicating if the date belongs to a leap year             |

**Note**

You can use `DatetimeIndex.isocalendar().week` to access week of year date information.

Furthermore, if you have a `Series` with datetimelike values, then you can access these properties via the `.dt` accessor, as detailed in the section on `.dt` accessors.

You may obtain the year, week and day components of the ISO year from the ISO 8601 standard:


In [82]:
idx = pd.date_range(start="2019-12-29", freq="D", periods=4)
idx.isocalendar()

,year,week,day
2019-12-29,2019,52,7
2019-12-30,2020,1,1
2019-12-31,2020,1,2
2020-01-01,2020,1,3


In [83]:
idx.to_series().dt.isocalendar()

,year,week,day
2019-12-29,2019,52,7
2019-12-30,2020,1,1
2019-12-31,2020,1,2
2020-01-01,2020,1,3


### Shifting / Lagging
One may want to shift or lag the values in a time series back and forward in time. The method for this is `shift()`, which is available on all of the pandas objects.

In [84]:
ts = pd.Series(range(len(rng)), index=rng)
ts = ts[:5]
ts.shift(1)

2011-01-31    NaN
2011-02-28    0.0
2011-03-31    1.0
2011-04-29    2.0
2011-05-31    3.0
Freq: BME, dtype: float64

The `shift` method accepts a `freq` argument which can accept a `DateOffset` class or other `timedelta`-like object or also an **offset alias**.

When `freq` is specified, `shift` method changes all the dates in the index rather than changing the alignment of the data and the index:


In [85]:
ts.shift(5, freq="D")

2011-02-05    0
2011-03-05    1
2011-04-05    2
2011-05-04    3
2011-06-05    4
dtype: int64

In [86]:
ts.shift(5, freq=pd.offsets.BDay())

2011-02-07    0
2011-03-07    1
2011-04-07    2
2011-05-06    3
2011-06-07    4
dtype: int64

In [87]:
ts.shift(5, freq="BME")
# Note that with when freq is specified, the leading entry is no longer NaN because the data is not being realigned.

2011-06-30    0
2011-07-29    1
2011-08-31    2
2011-09-30    3
2011-10-31    4
Freq: BME, dtype: int64

## Resampling

pandas has a simple, powerful, and efficient functionality for performing resampling operations during frequency conversion (e.g., converting secondly data into 5-minutely data). This is extremely common in, but not limited to, financial applications.

**`resample()`** is a time-based groupby, followed by a reduction method on each of its groups. See some cookbook examples for some advanced strategies.

The `resample()` method can be used directly from `DataFrameGroupBy` objects, see the groupby docs.

### Basics


In [88]:
rng = pd.date_range('1/1/2012', periods =100, freq = 's')
ts = pd.Series(np.random.randint(0, 500, len(rng)), index=rng)
ts.resample('5Min').sum()

2012-01-01    25483
Freq: 5min, dtype: int64

The `resample` function is very flexible and allows you to specify many different parameters to control the frequency conversion and resampling operation.

Any built-in method available via **GroupBy** is available as a method of the returned object, including `sum`, `mean`, `std`, `sem`, `max`, `min`, `median`, `first`, `last`, `ohlc`:


In [89]:
ts.resample('5Min').mean()

2012-01-01    254.83
Freq: 5min, dtype: float64

In [90]:
ts.resample("5Min").ohlc()

,open,high,low,close
2012-01-01,238,496,3,244


In [91]:
ts.resample("5Min").max()

2012-01-01    496
Freq: 5min, dtype: int64

In [92]:
# For downsampling, closed can be set to ‘left’ or ‘right’ to specify which end of the interval is closed:
ts.resample('5Min', closed = 'right').mean()

2011-12-31 23:55:00    238.0
2012-01-01 00:00:00    255.0
Freq: 5min, dtype: float64

In [93]:
ts.resample('5Min', closed = 'left').mean()

2012-01-01    254.83
Freq: 5min, dtype: float64

In [94]:
# Parameters like label are used to manipulate the resulting labels. label specifies whether the result is labeled with the beginning or the end of the interval.
ts.resample("5Min").mean()  # by default label='left'

2012-01-01    254.83
Freq: 5min, dtype: float64

In [95]:
ts.resample("5Min", label="left").mean()

2012-01-01    254.83
Freq: 5min, dtype: float64

##### ⚠️ Warning

The default values for `label` and `closed` is ‘**left**’ for all frequency offsets except for ‘ME’, ‘YE’, ‘QE’, ‘BME’, ‘BYE’, ‘BQE’, and ‘W’ which all have a default of ‘right’.

This might unintendedly lead to looking ahead, where the value for a later time is pulled back to a previous time as in the following example with the **`BusinessDay`** frequency:


In [96]:
s = pd.date_range("2000-01-01", "2000-01-05").to_series()
s.iloc[2] = pd.NaT
s.dt.day_name()

2000-01-01     Saturday
2000-01-02       Sunday
2000-01-03          NaN
2000-01-04      Tuesday
2000-01-05    Wednesday
Freq: D, dtype: str

In [97]:
s.resample('B', label = 'right', closed = 'right').last().dt.day_name()

2000-01-03       Sunday
2000-01-04      Tuesday
2000-01-05    Wednesday
2000-01-06          NaN
Freq: B, dtype: str

The `axis` parameter can be set to 0 or 1 and allows you to resample the specified axis for a `DataFrame`.

`kind` can be set to ‘timestamp’ or ‘period’ to convert the resulting index to/from timestamp and time span representations. By default `resample` retains the input representation.

`convention` can be set to ‘start’ or ‘end’ when resampling period data (detail below). It specifies how low frequency periods are converted to higher frequency periods.

### Upsampling

For upsampling, you can specify a way to upsample and the `limit` parameter to interpolate over the gaps that are created:

In [98]:
# from secondly to every 250 milliseconds
ts[:2].resample("250ms").asfreq()

2012-01-01 00:00:00.000    238.0
2012-01-01 00:00:00.250      NaN
2012-01-01 00:00:00.500      NaN
2012-01-01 00:00:00.750      NaN
2012-01-01 00:00:01.000     89.0
Freq: 250ms, dtype: float64

In [99]:
ts[:2].resample("250ms").ffill()

2012-01-01 00:00:00.000    238
2012-01-01 00:00:00.250    238
2012-01-01 00:00:00.500    238
2012-01-01 00:00:00.750    238
2012-01-01 00:00:01.000     89
Freq: 250ms, dtype: int64

In [100]:
ts[:2].resample("250ms").ffill(limit=2)

2012-01-01 00:00:00.000    238.0
2012-01-01 00:00:00.250    238.0
2012-01-01 00:00:00.500    238.0
2012-01-01 00:00:00.750      NaN
2012-01-01 00:00:01.000     89.0
Freq: 250ms, dtype: float64

### Aggregation

The `resample()` method returns a `pandas.api.typing.Resampler` instance. Similar to the **aggregating API**, **groupby API**, and the **window API**, a `Resampler` can be selectively resampled.

Resampling a `DataFrame`, the default will be to act on all columns with the same function.


In [101]:
df = pd.DataFrame(
    np.random.randn(1000, 3),
    index=pd.date_range("1/1/2012", freq="s", periods=1000),
    columns=["A", "B", "C"],
)
r = df.resample("3min")
r.mean()

,A,B,C
2012-01-01 00:00:00,-0.035993,-0.043519,-0.085054
2012-01-01 00:03:00,-0.064879,-0.023946,-0.063234
2012-01-01 00:06:00,0.021493,-0.005994,0.012908
2012-01-01 00:09:00,-0.146234,-0.037523,0.002835
2012-01-01 00:12:00,-0.054029,-0.114271,0.067996
2012-01-01 00:15:00,0.315029,-0.021707,-0.139493


In [102]:
# We can select a specific column or columns using standard getitem.
r["A"].mean()

2012-01-01 00:00:00   -0.035993
2012-01-01 00:03:00   -0.064879
2012-01-01 00:06:00    0.021493
2012-01-01 00:09:00   -0.146234
2012-01-01 00:12:00   -0.054029
2012-01-01 00:15:00    0.315029
Freq: 3min, Name: A, dtype: float64

In [103]:
r[["A", "B"]].mean()

,A,B
2012-01-01 00:00:00,-0.035993,-0.043519
2012-01-01 00:03:00,-0.064879,-0.023946
2012-01-01 00:06:00,0.021493,-0.005994
2012-01-01 00:09:00,-0.146234,-0.037523
2012-01-01 00:12:00,-0.054029,-0.114271
2012-01-01 00:15:00,0.315029,-0.021707


In [104]:
# You can pass a list or dict of functions to do aggregation with, outputting a DataFrame:
r["A"].agg(["sum", "mean", "std"])

,sum,mean,std
2012-01-01 00:00:00,-6.478695,-0.035993,1.036928
2012-01-01 00:03:00,-11.678289,-0.064879,1.070931
2012-01-01 00:06:00,3.868656,0.021493,1.032657
2012-01-01 00:09:00,-26.322178,-0.146234,1.089233
2012-01-01 00:12:00,-9.725249,-0.054029,0.928021
2012-01-01 00:15:00,31.502862,0.315029,1.104634


In [105]:
# On a resampled DataFrame, you can pass a list of functions to apply to each column, which produces an aggregated result with a hierarchical index:
r.agg(["sum", "mean"])

A                    B                    C  \
                           sum      mean        sum      mean        sum   
2012-01-01 00:00:00  -6.478695 -0.035993  -7.833418 -0.043519 -15.309698   
2012-01-01 00:03:00 -11.678289 -0.064879  -4.310202 -0.023946 -11.382101   
2012-01-01 00:06:00   3.868656  0.021493  -1.078893 -0.005994   2.323376   
2012-01-01 00:09:00 -26.322178 -0.146234  -6.754078 -0.037523   0.510259   
2012-01-01 00:12:00  -9.725249 -0.054029 -20.568869 -0.114271  12.239239   
2012-01-01 00:15:00  31.502862  0.315029  -2.170724 -0.021707 -13.949337   

                               
                         mean  
2012-01-01 00:00:00 -0.085054  
2012-01-01 00:03:00 -0.063234  
2012-01-01 00:06:00  0.012908  
2012-01-01 00:09:00  0.002835  
2012-01-01 00:12:00  0.067996  
2012-01-01 00:15:00 -0.139493

In [106]:
# By passing a dict to aggregate you can apply a different aggregation to the columns of a DataFrame:
r.agg({"A": "sum", "B": lambda x: np.std(x, ddof=1)})

,A,B
2012-01-01 00:00:00,-6.478695,0.992087
2012-01-01 00:03:00,-11.678289,1.053500
2012-01-01 00:06:00,3.868656,0.907395
2012-01-01 00:09:00,-26.322178,0.999753
2012-01-01 00:12:00,-9.725249,1.019188
2012-01-01 00:15:00,31.502862,1.124285


In [107]:
# The function names can also be strings. In order for a string to be valid it must be implemented on the resampled object:
r.agg({"A": "sum", "B": "std"})

,A,B
2012-01-01 00:00:00,-6.478695,0.992087
2012-01-01 00:03:00,-11.678289,1.053500
2012-01-01 00:06:00,3.868656,0.907395
2012-01-01 00:09:00,-26.322178,0.999753
2012-01-01 00:12:00,-9.725249,1.019188
2012-01-01 00:15:00,31.502862,1.124285


In [108]:
# Furthermore, you can also specify multiple aggregation functions for each column separately.
r.agg({"A": ["sum", "std"], "B": ["mean", "std"]})

A                   B          
                           sum       std      mean       std
2012-01-01 00:00:00  -6.478695  1.036928 -0.043519  0.992087
2012-01-01 00:03:00 -11.678289  1.070931 -0.023946  1.053500
2012-01-01 00:06:00   3.868656  1.032657 -0.005994  0.907395
2012-01-01 00:09:00 -26.322178  1.089233 -0.037523  0.999753
2012-01-01 00:12:00  -9.725249  0.928021 -0.114271  1.019188
2012-01-01 00:15:00  31.502862  1.104634 -0.021707  1.124285

In [109]:
# If a DataFrame does not have a datetimelike index, but instead you want to resample based on datetimelike column in the frame, it can passed to the on keyword.
df = pd.DataFrame(
    {"date": pd.date_range("2015-01-01", freq="W", periods=5), "a": np.arange(5)},
    index=pd.MultiIndex.from_arrays(
        [[1, 2, 3, 4, 5], pd.date_range("2015-01-01", freq="W", periods=5)],
        names=["v", "d"],
    ),
)
df

,,date,a
v,d,,
1,2015-01-04,2015-01-04,0
2,2015-01-11,2015-01-11,1
3,2015-01-18,2015-01-18,2
4,2015-01-25,2015-01-25,3
5,2015-02-01,2015-02-01,4


In [110]:
df.resample("MS", on="date")[["a"]].sum()

,a
date,
2015-01-01,6
2015-02-01,4


In [111]:
# Similarly, if you instead want to resample by a datetimelike level of MultiIndex, its name or location can be passed to the level keyword.
df.resample("MS", level="d")[["a"]].sum()

,a
d,
2015-01-01,6
2015-02-01,4


## Time zone handling

pandas provides rich support for working with timestamps in different time zones using the `zoneinfo`, `pytz` and `dateutil` libraries or `datetime.timezone` objects from the standard library.

### Working with time zones
By default, pandas objects are time zone unaware:

In [112]:
rng = pd.date_range("3/6/2012 00:00", periods=15, freq="D")
rng.tz is None

True

To localize these dates to a time zone (assign a particular time zone to a naive date), you can use the `tz_localize` method or the `tz` keyword argument in **`date_range()`**, **`Timestamp`**, or **`DatetimeIndex`**. You can either pass `zoneinfo`, `pytz` or `dateutil` time zone objects or Olson time zone database strings. Olson time zone strings will return `pytz` time zone objects by default. To return `dateutil` time zone objects, append `dateutil/` before the string.

* For `zoneinfo`, a list of available timezones are available from `zoneinfo.available_timezones()`.
* In `pytz` you can find a list of common (and less common) time zones using `pytz.all_timezones`.
* `dateutil` uses the OS time zones so there isn't a fixed list available. For common zones, the names are the same as `pytz` and `zoneinfo`.

In [113]:
import dateutil

# pytz
rng_pytz = pd.date_range("3/6/2012 00:00", periods=3, freq="D", tz="Europe/London")
rng_pytz.tz

zoneinfo.ZoneInfo(key='Europe/London')

In [114]:
# dateutil
rng_dateutil = pd.date_range("3/6/2012 00:00", periods=3, freq="D")
rng_dateutil = rng_dateutil.tz_localize("dateutil/Europe/London")
rng_dateutil.tz

tzfile('/usr/share/zoneinfo/Europe/London')

In [115]:
# dateutil - utc special case
rng_utc = pd.date_range(
    "3/6/2012 00:00",
    periods=3,
    freq="D",
    tz=dateutil.tz.tzutc(),
)
rng_utc.tz

tzutc()

In [116]:
# datetime.timezone
rng_utc = pd.date_range(
    "3/6/2012 00:00",
    periods=3,
    freq="D",
    tz=datetime.timezone.utc,
)
rng_utc.tz

datetime.timezone.utc

Note that the `UTC` time zone is a special case in `dateutil` and should be constructed explicitly as an instance of `dateutil.tz.tzutc`. You can also construct other time zones objects explicitly first.

> **Note:** pandas 3 does not install `pytz` for you. If the next cell gives `ModuleNotFoundError: No module named 'pytz'`, run `pip install pytz` once, then run the cell again.

In [117]:
import pytz

# pytz
tz_pytz = pytz.timezone("Europe/London")
rng_pytz = pd.date_range("3/6/2012 00:00", periods=3, freq="D")
rng_pytz = rng_pytz.tz_localize(tz_pytz)
rng_pytz.tz == tz_pytz

True

In [118]:
# dateutil
tz_dateutil = dateutil.tz.gettz("Europe/London")
rng_dateutil = pd.date_range("3/6/2012 00:00", periods=3, freq="D", tz=tz_dateutil)
rng_dateutil.tz == tz_dateutil

True

To convert a time zone aware pandas object from one time zone to another, you can use the `tz_convert` method.

In [119]:
rng_pytz.tz_convert("US/Eastern")

DatetimeIndex(['2012-03-05 19:00:00-05:00', '2012-03-06 19:00:00-05:00',
               '2012-03-07 19:00:00-05:00'],
              dtype='datetime64[us, US/Eastern]', freq=None)

**Note**

When using `pytz` time zones, **`DatetimeIndex`** will construct a different time zone object than a **`Timestamp`** for the same time zone input. A **`DatetimeIndex`** can hold a collection of **`Timestamp`** objects that may have different UTC offsets and cannot be succinctly represented by one `pytz` time zone instance while one **`Timestamp`** represents one point in time with a specific UTC offset.

In [120]:
dti = pd.date_range("2019-01-01", periods=3, freq="D", tz="US/Pacific")
dti.tz

zoneinfo.ZoneInfo(key='US/Pacific')

In [121]:
ts = pd.Timestamp("2019-01-01", tz="US/Pacific")
ts.tz

zoneinfo.ZoneInfo(key='US/Pacific')

##### ⚠️ Warning

* Be wary of conversions between libraries. For some time zones, `pytz` and `dateutil` have different definitions of the zone. This is more of a problem for unusual time zones than for 'standard' zones like `US/Eastern`.
* Be aware that a time zone definition across versions of time zone libraries may not be considered equal. This may cause problems when working with stored data that is localized using one version and operated on with a different version.
* For `pytz` time zones, it is incorrect to pass a time zone object directly into the `datetime.datetime` constructor (e.g., `datetime.datetime(2011, 1, 1, tzinfo=pytz.timezone('US/Eastern'))`). Instead, the datetime needs to be localized using the `localize` method on the `pytz` time zone object.
* Be aware that for times in the future, correct conversion between time zones (and UTC) cannot be guaranteed by any time zone library because a timezone's offset from UTC may be changed by the respective government.
* If you are using dates beyond 2038-01-18 with `pytz`, due to current deficiencies in the underlying libraries caused by the year 2038 problem, daylight saving time (DST) adjustments to timezone aware dates will not be applied.

Under the hood, all timestamps are stored in UTC. Values from a time zone aware **`DatetimeIndex`** or **`Timestamp`** will have their fields (day, hour, minute, etc.) localized to the time zone. However, timestamps with the same UTC value are still considered to be equal even if they are in different time zones:

In [122]:
rng_eastern = rng_utc.tz_convert("US/Eastern")
rng_berlin = rng_utc.tz_convert("Europe/Berlin")

rng_eastern[2]

Timestamp('2012-03-07 19:00:00-0500', tz='US/Eastern')

In [123]:
rng_berlin[2]

Timestamp('2012-03-08 01:00:00+0100', tz='Europe/Berlin')

In [124]:
rng_eastern[2] == rng_berlin[2]

True

In [125]:
# Operations between Series in different time zones will yield UTC Series, aligning the data on the UTC timestamps:
ts_utc = pd.Series(range(3), pd.date_range("20130101", periods=3, tz="UTC"))
eastern = ts_utc.tz_convert("US/Eastern")
berlin = ts_utc.tz_convert("Europe/Berlin")
result = eastern + berlin
result

2013-01-01 00:00:00+00:00    0
2013-01-02 00:00:00+00:00    2
2013-01-03 00:00:00+00:00    4
dtype: int64

In [126]:
result.index

DatetimeIndex(['2013-01-01 00:00:00+00:00', '2013-01-02 00:00:00+00:00',
               '2013-01-03 00:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq=None)

To remove time zone information, use `tz_localize(None)` or `tz_convert(None)`. `tz_localize(None)` will remove the time zone yielding the local time representation. `tz_convert(None)` will remove the time zone after converting to UTC time.

In [127]:
didx = pd.date_range(start="2014-08-01 09:00", freq="h", periods=3, tz="US/Eastern")
didx

DatetimeIndex(['2014-08-01 09:00:00-04:00', '2014-08-01 10:00:00-04:00',
               '2014-08-01 11:00:00-04:00'],
              dtype='datetime64[us, US/Eastern]', freq='h')

In [128]:
didx.tz_localize(None)

DatetimeIndex(['2014-08-01 09:00:00', '2014-08-01 10:00:00',
               '2014-08-01 11:00:00'],
              dtype='datetime64[us]', freq=None)

In [129]:
didx.tz_convert(None)

DatetimeIndex(['2014-08-01 13:00:00', '2014-08-01 14:00:00',
               '2014-08-01 15:00:00'],
              dtype='datetime64[us]', freq='h')

In [130]:
# tz_convert(None) is identical to tz_convert('UTC').tz_localize(None)
didx.tz_convert("UTC").tz_localize(None)

DatetimeIndex(['2014-08-01 13:00:00', '2014-08-01 14:00:00',
               '2014-08-01 15:00:00'],
              dtype='datetime64[us]', freq=None)

# Windowing operations
📄 https://pandas.pydata.org/docs/user_guide/window.html

pandas contains a compact set of APIs for performing windowing operations - an operation that performs an aggregation over a sliding partition of values. The API functions similarly to the `groupby` API in that **`Series`** and **`DataFrame`** call the windowing method with necessary parameters and then subsequently call the aggregation function.

In [131]:
s = pd.Series(range(5))
s.rolling(window=2).sum()

0    NaN
1    1.0
2    3.0
3    5.0
4    7.0
dtype: float64

In [132]:
# The windows are comprised by looking back the length of the window from the current observation.
# The result above can be derived by taking the sum of the following windowed partitions of data:
for window in s.rolling(window=2):
    print(window)

0    0
dtype: int64
0    0
1    1
dtype: int64
1    1
2    2
dtype: int64
2    2
3    3
dtype: int64
3    3
4    4
dtype: int64


### Overview

pandas supports 4 types of windowing operations:

1. Rolling window: Generic fixed or variable sliding window over the values.
2. Weighted window: Weighted, non-rectangular window supplied by the `scipy.signal` library.
3. Expanding window: Accumulating window over the values.
4. Exponentially Weighted window: Accumulating and exponentially weighted window over the values.

| Concept | Method | Returned Object | Supports time-based windows | Supports chained groupby | Supports table method | Supports online operations |
| --- | --- | --- | --- | --- | --- | --- |
| Rolling window | `rolling` | `pandas.typing.api.Rolling` | Yes | Yes | Yes (as of version 1.3) | No |
| Weighted window | `rolling` | `pandas.typing.api.Window` | No | No | No | No |
| Expanding window | `expanding` | `pandas.typing.api.Expanding` | No | Yes | Yes (as of version 1.3) | No |
| Exponentially Weighted window | `ewm` | `pandas.typing.api.ExponentialMovingWindow` | No | Yes (as of version 1.2) | No | Yes (as of version 1.3) |

As noted above, some operations support specifying a window based on a time offset:

In [133]:
s = pd.Series(range(5), index=pd.date_range('2020-01-01', periods=5, freq='1D'))
s.rolling(window='2D').sum()

2020-01-01    0.0
2020-01-02    1.0
2020-01-03    3.0
2020-01-04    5.0
2020-01-05    7.0
Freq: D, dtype: float64

In [134]:
# Additionally, some methods support chaining a groupby operation with a windowing operation
# which will first group the data by the specified keys and then perform a windowing operation per group.
df = pd.DataFrame({'A': ['a', 'b', 'a', 'b', 'a'], 'B': range(5)})
df.groupby('A').expanding().sum()

B
A       
a 0  0.0
  2  2.0
  4  6.0
b 1  1.0
  3  4.0

**Note**

Windowing operations currently only support numeric data (integer and float) and will always return `float64` values.

##### ⚠️ Warning

Some windowing aggregation, `mean`, `sum`, `var` and `std` methods may suffer from numerical imprecision due to the underlying windowing algorithms accumulating sums. When values differ with magnitude `1/np.finfo(np.double).eps` (approximately $4.5 \times 10^{15}$), this results in truncation. It must be noted, that large values may have an impact on windows, which do not include these values. Kahan summation is used to compute the rolling sums to preserve accuracy as much as possible.

*(The docs' `method="table"` and `online` examples are skipped here: they need the `numba` package and are not part of the plan.)*

All windowing operations support a `min_periods` argument that dictates the minimum amount of non-`np.nan` values a window must have; otherwise, the resulting value is `np.nan`. `min_periods` defaults to 1 for time-based windows and `window` for fixed windows

In [135]:
s = pd.Series([np.nan, 1, 2, np.nan, np.nan, 3])
s.rolling(window=3, min_periods=1).sum()

0    NaN
1    1.0
2    3.0
3    3.0
4    2.0
5    3.0
dtype: float64

In [136]:
s.rolling(window=3, min_periods=2).sum()

0    NaN
1    NaN
2    3.0
3    3.0
4    NaN
5    NaN
dtype: float64

In [137]:
# Equivalent to min_periods=3
s.rolling(window=3, min_periods=None).sum()

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
dtype: float64

In [138]:
# Additionally, all windowing operations supports the aggregate method for returning a result of multiple aggregations applied to a window.
df = pd.DataFrame({"A": range(5), "B": range(10, 15)})
df.expanding().agg(["sum", "mean", "std"])

A                    B                
    sum mean       std   sum  mean       std
0   0.0  0.0       NaN  10.0  10.0       NaN
1   1.0  0.5  0.707107  21.0  10.5  0.707107
2   3.0  1.0  1.000000  33.0  11.0  1.000000
3   6.0  1.5  1.290994  46.0  11.5  1.290994
4  10.0  2.0  1.581139  60.0  12.0  1.581139

## Rolling window

Generic rolling windows support specifying windows as a fixed number of observations or variable number of observations based on an offset. If a time based offset is provided, the corresponding time based index must be monotonic.

In [139]:
times = ['2020-01-01', '2020-01-03', '2020-01-04', '2020-01-05', '2020-01-29']
s = pd.Series(range(5), index=pd.DatetimeIndex(times))
s

2020-01-01    0
2020-01-03    1
2020-01-04    2
2020-01-05    3
2020-01-29    4
dtype: int64

In [140]:
# Window with 2 observations
s.rolling(window=2).sum()

2020-01-01    NaN
2020-01-03    1.0
2020-01-04    3.0
2020-01-05    5.0
2020-01-29    7.0
dtype: float64

In [141]:
# Window with 2 days worth of observations
s.rolling(window='2D').sum()

2020-01-01    0.0
2020-01-03    1.0
2020-01-04    3.0
2020-01-05    5.0
2020-01-29    4.0
dtype: float64

### Centering windows

By default the labels are set to the right edge of the window, but a `center` keyword is available so the labels can be set at the center.

In [142]:
s = pd.Series(range(10))
s.rolling(window=5).mean()

0    NaN
1    NaN
2    NaN
3    NaN
4    2.0
5    3.0
6    4.0
7    5.0
8    6.0
9    7.0
dtype: float64

In [143]:
s.rolling(window=5, center=True).mean()

0    NaN
1    NaN
2    2.0
3    3.0
4    4.0
5    5.0
6    6.0
7    7.0
8    NaN
9    NaN
dtype: float64

In [144]:
# This can also be applied to datetime-like indices.
df = pd.DataFrame(
    {"A": [0, 1, 2, 3, 4]}, index=pd.date_range("2020", periods=5, freq="1D")
)
df

,A
2020-01-01,0
2020-01-02,1
2020-01-03,2
2020-01-04,3
2020-01-05,4


In [145]:
df.rolling("2D", center=False).mean()

,A
2020-01-01,0.0
2020-01-02,0.5
2020-01-03,1.5
2020-01-04,2.5
2020-01-05,3.5


In [146]:
df.rolling("2D", center=True).mean()

,A
2020-01-01,0.5
2020-01-02,1.5
2020-01-03,2.5
2020-01-04,3.5
2020-01-05,4.0


## Expanding window

An expanding window yields the value of an aggregation statistic with all the data available up to that point in time. Since these calculations are a special case of rolling statistics, they are implemented in pandas such that the following two calls are equivalent:

In [147]:
df = pd.DataFrame(range(5))
df.rolling(window=len(df), min_periods=1).mean()

,0
0,0.0
1,0.5
2,1.0
3,1.5
4,2.0


In [148]:
df.expanding(min_periods=1).mean()

,0
0,0.0
1,0.5
2,1.0
3,1.5
4,2.0


## Exponentially weighted window

An exponentially weighted window is similar to an expanding window but with each prior point being exponentially weighted down relative to the current point.

In general, a weighted moving average is calculated as

$$y_t = \frac{\sum_{i=0}^t w_i x_{t-i}}{\sum_{i=0}^t w_i},$$

where $x_t$ is the input, $y_t$ is the result and the $w_i$ are the weights.

The EW functions support two variants of exponential weights. The default, `adjust=True`, uses the weights $w_i = (1 - \alpha)^i$ which gives

$$y_t = \frac{x_t + (1 - \alpha)x_{t-1} + (1 - \alpha)^2 x_{t-2} + ... + (1 - \alpha)^t x_{0}}{1 + (1 - \alpha) + (1 - \alpha)^2 + ... + (1 - \alpha)^t}$$

When `adjust=False` is specified, moving averages are calculated as

$$\begin{aligned} y_0 &= x_0 \\ y_t &= (1 - \alpha) y_{t-1} + \alpha x_t, \end{aligned}$$

which is equivalent to using weights

$$w_i = \begin{cases} \alpha (1 - \alpha)^i & \text{if } i < t \\ (1 - \alpha)^i & \text{if } i = t. \end{cases}$$

**Note:** These equations are sometimes written in terms of $\alpha' = 1 - \alpha$, e.g. $y_t = \alpha' y_{t-1} + (1 - \alpha') x_t.$

The difference between the above two variants arises because we are dealing with series which have finite history. For a series of infinite history the two variants are the same. When `adjust=False`, we have $y_0 = x_0$ and $y_t = \alpha x_t + (1 - \alpha) y_{t-1}$. Therefore, there is an assumption that $x_0$ is not an ordinary value but rather an exponentially weighted moment of the infinite series up to that point.

One must have $0 < \alpha \leq 1$, and while it is possible to pass $\alpha$ directly, it's often easier to think about either the **span**, **center of mass (com)** or **half-life** of an EW moment:

$$\alpha = \begin{cases} \frac{2}{s + 1}, & \text{for span}\ s \geq 1\\ \frac{1}{1 + c}, & \text{for center of mass}\ c \geq 0\\ 1 - e^{\frac{\log 0.5}{h}}, & \text{for half-life}\ h > 0 \end{cases}$$

One must specify precisely one of **span**, **center of mass**, **half-life** and **alpha** to the EW functions:

* **Span** corresponds to what is commonly called an "N-day EW moving average".
* **Center of mass** has a more physical interpretation and can be thought of in terms of span: $c = (s - 1) / 2$.
* **Half-life** is the period of time for the exponential weight to reduce to one half.
* **Alpha** specifies the smoothing factor directly.

You can also specify `halflife` in terms of a timedelta convertible unit to specify the amount of time it takes for an observation to decay to half its value when also specifying a sequence of `times`.

In [149]:
df = pd.DataFrame({"B": [0, 1, 2, np.nan, 4]})
df

,B
0,0.0
1,1.0
2,2.0
3,NaN
4,4.0


In [150]:
times = ["2020-01-01", "2020-01-03", "2020-01-10", "2020-01-15", "2020-01-17"]
df.ewm(halflife="4 days", times=pd.DatetimeIndex(times)).mean()

,B
0,0.000000
1,0.585786
2,1.523889
3,1.523889
4,3.233686


The following formula is used to compute exponentially weighted mean with an input vector of times:

$$y_t = \frac{\sum_{i=0}^t 0.5^\frac{t_{t} - t_{i}}{\lambda} x_{t-i}}{\sum_{i=0}^t 0.5^\frac{t_{t} - t_{i}}{\lambda}},$$

ExponentialMovingWindow also has an `ignore_na` argument, which determines how intermediate null values affect the calculation of the weights. When `ignore_na=False` (the default), weights are calculated based on absolute positions, so that intermediate null values affect the result. When `ignore_na=True`, weights are calculated by ignoring intermediate null values. For example, assuming `adjust=True`, if `ignore_na=False`, the weighted average of `3, NaN, 5` would be calculated as

$$\frac{(1-\alpha)^2 \cdot 3 + 1 \cdot 5}{(1-\alpha)^2 + 1}.$$

Whereas if `ignore_na=True`, the weighted average would be calculated as

$$\frac{(1-\alpha) \cdot 3 + 1 \cdot 5}{(1-\alpha) + 1}.$$

The `var()`, `std()`, and `cov()` functions have a `bias` argument, specifying whether the result should contain biased or unbiased statistics. For example, if `bias=True`, `ewmvar(x)` is calculated as `ewmvar(x) = ewma(x**2) - ewma(x)**2`; whereas if `bias=False` (the default), the biased variance statistics are scaled by debiasing factors

$$\frac{\left(\sum_{i=0}^t w_i\right)^2}{\left(\sum_{i=0}^t w_i\right)^2 - \sum_{i=0}^t w_i^2}.$$

(For $w_i = 1$, this reduces to the usual $N / (N - 1)$ factor, with $N = t + 1$.)